# Lab4: ANN fo classifying satellite images  

#### Brief explanation:

The task can be posed as a function that maps an input satellite image patch to a specific category. The input X is a satellite image patch cut off from bigger image, while the output Y is its tag.

The goal is to train ANN to learn from train examples and correctly classify new patches.

#### Preparing data

First, Sentinel-2 satellite imagery was downloaded and automatically divided into 100 × 100 pixel patches. The patches were then randomly assigned to training and testing datasets and labeled as one of two classes: city or forest.

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "train").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "Lab4"

TRAIN_DIR = NOTEBOOK_DIR / "train"
TEST_DIR = NOTEBOOK_DIR / "test"
CLASS_NAMES = {"city": 0, "forest": 1}
CLASS_LABELS = ["city", "forest"]

In [2]:
def list_patch_files(split_dir: Path) -> list[Path]:
    files = []
    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Error: {class_dir}")
        files.extend(sorted(class_dir.glob("*.tif")))
    return files

inventory_rows = []
for split_name, split_dir in [("train", TRAIN_DIR), ("test", TEST_DIR)]:
    for class_name, class_id in CLASS_NAMES.items():
        paths = sorted((split_dir / class_name).glob("*.tif"))
        inventory_rows.append({
            "split": split_name,
            "class": class_name,
            "patches": len(paths),
        })

inventory = pd.DataFrame(inventory_rows)
display(inventory)

,split,class,patches
0,train,city,158
1,train,forest,658
2,test,city,40
3,test,forest,163


#### Baseline ANN

Baseline ANN is a basic neural network which architecture typically consists of three components: an input layer, hidden layer and an output layer. The network learns through forward propagation and backpropagation, adjusting neuron weights to minimize prediction errors.


In [3]:
def read_patch(path: Path) -> np.ndarray:
    with rasterio.open(path) as src:
        image = src.read().astype(np.float32)
        return image

def patch_to_features(path: Path) -> np.ndarray:
    image = read_patch(path)
    channels, height, width = image.shape
    if (channels, height, width) != (3, 100, 100):
        raise ValueError(f"{path.name}: oczekiwano (3, 100, 100), jest {image.shape}")
    return image.reshape(-1).astype(np.float32)

def build_dataset(split_dir: Path) -> tuple[np.ndarray, np.ndarray, list[Path]]:
    paths = list_patch_files(split_dir)
    features = np.vstack([patch_to_features(path) for path in paths])
    labels = np.array([CLASS_NAMES[path.parent.name] for path in paths], dtype=np.int64)
    return features, labels, paths

X_train, y_train, train_paths = build_dataset(TRAIN_DIR)
X_test, y_test, test_paths = build_dataset(TEST_DIR)

train_invalid = ~np.isfinite(X_train)
test_invalid = ~np.isfinite(X_test)
X_train_for_median = X_train.copy()
X_train_for_median[train_invalid] = np.nan
feature_medians = np.nanmedian(X_train_for_median, axis=0)
if not np.all(np.isfinite(feature_medians)):
    raise ValueError("Co najmniej jedna cecha jest pusta w całym train.")
X_train = np.where(train_invalid, feature_medians, X_train).astype(np.float32)
X_test = np.where(test_invalid, feature_medians, X_test).astype(np.float32)


In [4]:
BASELINE_NAME = "16 ReLU + Adam"
baseline_config = {
    "hidden_layer_sizes": (16,),
    "activation": "relu",
    "solver": "adam",
    "alpha": 1e-4,
    "learning_rate_init": 1e-3,
    "max_iter": 300,
}

def fit_and_evaluate(name: str, config: dict):
    config_for_mlp = {
        key: value for key, value in config.items()
        if key != "name"
    }
    config_for_mlp.update({
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 25,
        "random_state": RANDOM_STATE,
    })

    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(**config_for_mlp),
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    mlp = model.named_steps["mlpclassifier"]

    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "iterations": mlp.n_iter_,
        "final_loss": mlp.loss_,
    }
    return model, row, mlp.loss_curve_

baseline_model, baseline_row, baseline_history = fit_and_evaluate(
    BASELINE_NAME,
    baseline_config,
)

models = {BASELINE_NAME: baseline_model}
histories = {BASELINE_NAME: baseline_history}
metrics_rows = [baseline_row]

baseline_results = pd.DataFrame([baseline_row])
display(
    baseline_results.style.format({
        column: "{:.4f}"
        for column in baseline_results.columns
        if column not in {"model", "iterations"}
    })
)

,model,accuracy,iterations,final_loss
0,16 ReLU + Adam,0.9704,27,0.0153


#### Another variants

In [12]:
architecture_variants = [
    {
        "name": "64 ReLU + Adam",
        "hidden_layer_sizes": (64,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 300,
    },
    {
        "name": "64-32 ReLU + Adam",
        "hidden_layer_sizes": (64, 32),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 300,
    },
    {
        "name": "64-32-16 ReLU + Adam",
        "hidden_layer_sizes": (64, 32, 16),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 300,
    },
    {
        "name": "32 tanh + Adam",
        "hidden_layer_sizes": (32,),
        "activation": "tanh",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 300,
    },
    {
        "name": "32 ReLU + SGD",
        "hidden_layer_sizes": (32,),
        "activation": "relu",
        "solver": "sgd",
        "alpha": 1e-4,
        "learning_rate_init": 1e-2,
        "learning_rate": "adaptive",
        "momentum": 0.9,
        "max_iter": 400,
    },
]

# Zerowanie wyników, aby ponowne uruchomienie komórki nie dublowało wariantów.
models = {BASELINE_NAME: baseline_model}
histories = {BASELINE_NAME: baseline_history}
metrics_rows = [baseline_row]

for config in architecture_variants:
    model, row, loss_curve = fit_and_evaluate(
        config["name"],
        config,
    )
    models[config["name"]] = model
    histories[config["name"]] = loss_curve
    metrics_rows.append(row)

results = (
    pd.DataFrame(metrics_rows)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

print("Comparison of different configurations:")
display(
    results.style.format({
        column: "{:.4f}"
        for column in results.columns
        if column not in {"model", "iterations"}
    })
)

Comparison of different configurations:


,model,accuracy,iterations,final_loss
0,64-32 ReLU + Adam,1.0000,35,0.0002
1,32 ReLU + SGD,0.9951,209,0.0020
2,64-32-16 ReLU + Adam,0.9901,32,0.0001
3,64 ReLU + Adam,0.9852,31,0.0001
4,16 ReLU + Adam,0.9704,27,0.0153
5,32 tanh + Adam,0.9704,27,0.1225


The ANN classified image patches as urban or forested areas, achieving 97.04% accuracy with the baseline configuration. The best variant, 64–32 ReLU + Adam, achieved 100% accuracy. Increasing the number of neurons and adding a hidden layer improved performance, whereas the 32 tanh + Adam variant did not outperform the baseline model.

Bayesian classifiers are far easier to use and require less tuning than ANN, thats why we prefer using them. Also they work well with small datasets which might be difficult with ANNs.